In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Target files
current_notebook = "index.ipynb"
notebook_files = [f for f in os.listdir('.') if f.endswith('.ipynb') and f != current_notebook]

# 2. AUTOMATIC REDIRECT TRICK
redirect_html = """
<script>
if (window.location.pathname.endswith('/voila') || window.location.pathname.endswith('/voila/')) {
    window.location.href = window.location.origin + '/voila/render/index.ipynb';
}
</script>
"""
display(HTML(redirect_html))

if sorted(notebook_files):
    # 3. Create the Content Display Area (Right Side)
    content_area = widgets.Output(layout=widgets.Layout(width='82%', height='85vh'))
    
    # Track the active button to style it later
    buttons = []
    
    # 4. Define what happens when a navigation button is clicked
    def on_nav_click(b):
        # Reset all button styles to default
        for btn in buttons:
            btn.button_style = ''
            btn.icon = ''
            
        # Highlight the selected button
        b.button_style = 'info'
        b.icon = 'check'
        
        # Clear the right side and load the new notebook iframe
        with content_area:
            content_area.clear_output(wait=True)
            iframe_url = f"/voila/render/{b.notebook_file}"
            
            # CSS script that forces text executions/streams to use a scroll box
            scroll_box_css = """
                .jp-OutputArea-output pre, 
                .output_text pre, 
                .output_stream, 
                .output_wrap pre {
                    max-height: 250px !important; 
                    overflow-y: auto !important; 
                    border: 1px solid #e2e8f0 !important; 
                    border-radius: 4px !important;
                    background-color: #f8fafc !important;
                    padding: 10px !important;
                }
            """.replace('\n', ' ')

            # The onload attribute injects the style block as soon as the iframe notebook finishes rendering
            iframe_html = f'''
            <iframe 
                src="{iframe_url}" 
                onload="
                    const style = this.contentDocument.createElement('style');
                    style.textContent = '{scroll_box_css}';
                    this.contentDocument.head.appendChild(style);
                "
                style="width:100%; height:85vh; border:none;">
            </iframe>
            '''
            display(HTML(iframe_html))

    # 5. Build the Vertical Sidebar (Left Side)
    sidebar_items = [widgets.HTML("<b style='font-size: 14px; padding: 5px; display: block;'>Notebooks</b>")]
    
    for nb_file in sorted(notebook_files):
        title = nb_file.replace('.ipynb', '').replace('_', ' ').title()
        
        # Create a clean button for each notebook
        btn = widgets.Button(
            description=title,
            layout=widgets.Layout(width='100%', margin='4px 0px'),
            alignment='left'
        )
        btn.notebook_file = nb_file  # Attach custom property to track the file
        btn.on_click(on_nav_click)
        
        buttons.append(btn)
        sidebar_items.append(btn)
        
    sidebar = widgets.VBox(sidebar_items, layout=widgets.Layout(width='18%', padding='10px', border='right: 1px solid #ddd'))
    
    # 6. Initialize the Dashboard with the first notebook loaded by default
    if buttons:
        on_nav_click(buttons[0])
        
    # 7. Combine Sidebar and Content side-by-side horizontally
    dashboard = widgets.HBox([sidebar, content_area], layout=widgets.Layout(width='100%'))
    display(dashboard)
    
else:
    print("No other .ipynb files found in this directory.")